# 03 — Linguistic loci analysis

This notebook measures **agreement and disagreement between two automatic annotators**. It does not determine which annotator is correct: there is no human gold standard.

The English text is a translation, so DE/EN contrasts are interpreted as corpus-version differences, not general language effects.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.metrics import bootstrap_agreement, cohen_kappa, permutation_agreement_baseline

sns.set_theme(style='whitegrid')


In [ ]:
def load_annotations(language: str) -> pd.DataFrame:
    path = ROOT / 'data' / 'processed' / f'annotations_{language}.csv'
    df = pd.read_csv(path)
    required = {'sent_id', 'upos_spacy', 'upos_llm_norm', 'parse_ok'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    valid = df.loc[df['parse_ok'].astype(str).str.lower().eq('true')].copy()
    valid['agree'] = valid['upos_spacy'].eq(valid['upos_llm_norm'])
    print(f'{language.upper()}: {valid.sent_id.nunique()}/{df.sent_id.nunique()} parse-ok sentences; {len(valid)} usable tokens')
    return valid

de = load_annotations('de')
en = load_annotations('en')


In [ ]:
def loci_table(df: pd.DataFrame, language: str) -> pd.DataFrame:
    table = (
        df.groupby('upos_spacy', observed=True)
        .agg(tokens=('agree', 'size'), agreement=('agree', 'mean'))
        .reset_index()
        .assign(
            language=language.upper(),
            agreement_pct=lambda x: 100 * x['agreement'],
            disagreement_pct=lambda x: 100 * (1 - x['agreement']),
        )
        .sort_values(['disagreement_pct', 'tokens'], ascending=[False, False])
    )
    return table

loci_de = loci_table(de, 'de')
loci_en = loci_table(en, 'en')
loci = pd.concat([loci_de, loci_en], ignore_index=True)
display(loci_de)
display(loci_en)


In [ ]:
plot_data = loci.loc[loci['tokens'] >= 10].copy()
plt.figure(figsize=(11, 6))
sns.barplot(
    data=plot_data,
    x='disagreement_pct', y='upos_spacy', hue='language',
    order=plot_data.groupby('upos_spacy')['disagreement_pct'].mean().sort_values().index,
)
plt.xlabel('Disagreement rate (%)')
plt.ylabel('SpaCy UPOS category')
plt.title('UPOS disagreement loci (categories with at least 10 tokens)')
plt.tight_layout()
plt.show()


In [ ]:
def top_confusions(df: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    return (
        df.loc[~df['agree']]
        .groupby(['upos_spacy', 'upos_llm_norm'], observed=True)
        .size()
        .rename('tokens')
        .reset_index()
        .sort_values('tokens', ascending=False)
        .head(n)
    )

print('DE confusion pairs')
display(top_confusions(de))
print('EN confusion pairs')
display(top_confusions(en))


In [ ]:
def overall_summary(df: pd.DataFrame, language: str) -> dict:
    observed, ci_low, ci_high = bootstrap_agreement(df['sent_id'], df['agree'])
    chance, chance_low, chance_high = permutation_agreement_baseline(
        df['upos_spacy'], df['upos_llm_norm']
    )
    return {
        'language': language.upper(),
        'tokens': len(df),
        'raw_agreement_pct': 100 * observed,
        'agreement_ci95_low_pct': 100 * ci_low,
        'agreement_ci95_high_pct': 100 * ci_high,
        'cohen_kappa': cohen_kappa(df['upos_spacy'], df['upos_llm_norm']),
        'permuted_chance_pct': 100 * chance,
        'permuted_chance_ci95_low_pct': 100 * chance_low,
        'permuted_chance_ci95_high_pct': 100 * chance_high,
    }

summary = pd.DataFrame([overall_summary(de, 'de'), overall_summary(en, 'en')])
display(summary.round(2))


In [ ]:
contrast = (
    loci_de[['upos_spacy', 'tokens', 'disagreement_pct']]
    .rename(columns={'tokens': 'tokens_de', 'disagreement_pct': 'disagreement_de_pct'})
    .merge(
        loci_en[['upos_spacy', 'tokens', 'disagreement_pct']]
        .rename(columns={'tokens': 'tokens_en', 'disagreement_pct': 'disagreement_en_pct'}),
        on='upos_spacy', how='outer'
    )
    .fillna(0)
    .assign(difference_de_minus_en_pct=lambda x: x['disagreement_de_pct'] - x['disagreement_en_pct'])
    .sort_values('difference_de_minus_en_pct', key=lambda x: x.abs(), ascending=False)
)
display(contrast)


## Reporting guardrails

- Report rates with their category token counts; do not over-interpret rare categories.
- Use the plot and confusion pairs as evidence of **disagreement loci**, not model errors.
- Treat the DE/EN contrast as a comparison of this German text and this English translation.
- Add qualitative token-level examples before writing the final interpretation.
